## 1. Imports

# 02 - ResNet50 Classifier

Fine-tunes a pre-trained **ResNet50** (ImageNet) for 36-class fruits & vegetables classification.

**Strategy**
- Phase 1: freeze the base model, train the new classification head only.
- Phase 2: unfreeze the top layers of ResNet50 (from index 140) and fine-tune end-to-end at a low LR.

**Input pipeline:** images are loaded as `[0, 1]` float32. A `Rescaling(255)` layer scales them back to
`[0, 255]`, then `resnet50.preprocess_input` converts to BGR and subtracts ImageNet channel means.

In [ ]:
%matplotlib inline
import sys
import json
import random
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {[g.name for g in gpus] if gpus else "none - CPU only"}')

## 2. Configuration

In [ ]:
ROOT       = Path('.')
TRAIN_ROOT = ROOT / 'train'
TEST_ROOT  = ROOT / 'test'
PLOTS_DIR  = ROOT / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
SEED         = 42

LR_HEAD      = 1e-3
LR_FINE      = 1e-5

EPOCHS_HEAD  = 10
EPOCHS_FINE  = 20

FINE_TUNE_AT = 140

DROPOUT      = 0.3

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

assert TRAIN_ROOT.exists(), 'train/ not found'
assert TEST_ROOT.exists(),  'test/ not found'

## 3. Data Helpers

In [3]:
def discover_classes(root: Path) -> dict:
    """
    Returns:
        {class_name: path}
    """
    classes = {}

    for cat_dir in sorted(root.iterdir()):
        if not cat_dir.is_dir():
            continue
        for cls_dir in sorted(cat_dir.iterdir()):
            if cls_dir.is_dir():
                classes[cls_dir.name.lower()] = cls_dir

    return classes


def build_file_list(root: Path, class_to_idx: dict):
    paths  = []
    labels = []

    for cat_dir in sorted(root.iterdir()):
        if not cat_dir.is_dir():
            continue
        for cls_dir in sorted(cat_dir.iterdir()):
            if not cls_dir.is_dir():
                continue
            cls = cls_dir.name.lower()
            if cls not in class_to_idx:
                continue
            idx = class_to_idx[cls]
            for f in cls_dir.iterdir():
                if f.suffix.lower() in IMG_EXTS:
                    paths.append(str(f))
                    labels.append(idx)

    return paths, labels


def make_tf_dataset(
    file_paths, int_labels, n_classes,
    img_size=(224, 224), batch_size=32,
    augment=False, shuffle=True, seed=42
):
    def load(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        return img, tf.one_hot(label, n_classes)

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.15)
        img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
        img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
        img = tf.clip_by_value(img, 0.0, 1.0)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((file_paths, int_labels))

    if shuffle:
        ds = ds.shuffle(len(file_paths), seed=seed)

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

## 4. Load Dataset

In [4]:
train_classes = discover_classes(TRAIN_ROOT)
CLASS_NAMES   = sorted(train_classes.keys())
N_CLASSES     = len(CLASS_NAMES)
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}

train_paths, train_labels = build_file_list(TRAIN_ROOT, CLASS_TO_IDX)
test_paths,  test_labels  = build_file_list(TEST_ROOT,  CLASS_TO_IDX)

train_ds = make_tf_dataset(
    train_paths, train_labels, N_CLASSES,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    augment=True, shuffle=True, seed=SEED
)

test_ds = make_tf_dataset(
    test_paths, test_labels, N_CLASSES,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    augment=False, shuffle=False
)

print(f'Classes     : {N_CLASSES}')
print(f'Train imgs  : {len(train_paths)}')
print(f'Test imgs   : {len(test_paths)}')

# Smoke test
for x, y in train_ds.take(1):
    print(f'Batch shape : {x.shape}')
    print(f'Labels      : {y.shape}')
    print(f'Pixel range : [{x.numpy().min():.3f}, {x.numpy().max():.3f}]')

Classes     : 36
Train imgs  : 3115
Test imgs   : 359
Batch shape : (32, 224, 224, 3)
Labels      : (32, 36)
Pixel range : [0.000, 1.000]


## 5. Build Model

In [5]:
# Load ResNet50 pretrained on ImageNet — without top classification layer
base_model = tf.keras.applications.ResNet50(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

print(f'Base model layers : {len(base_model.layers)}')



Base model layers : 175


In [6]:
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='input_image')

# Scale [0,1] → [0,255]
x = tf.keras.layers.Rescaling(scale=255.0, name='scale_to_255')(inputs)

# ResNet50 preprocessing — wrapped in Lambda to allow model saving
x = tf.keras.layers.Lambda(
    tf.keras.applications.resnet50.preprocess_input,
    name='preprocess'
)(x)

# Backbone (frozen)
x = base_model(x, training=False)

# Classification head
x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
x = tf.keras.layers.BatchNormalization(name='head_bn')(x)
x = tf.keras.layers.Dropout(DROPOUT, name='head_dropout')(x)
outputs = tf.keras.layers.Dense(N_CLASSES, activation='softmax', name='predictions')(x)

model = tf.keras.Model(inputs, outputs, name='resnet50_36cls')
model.summary()

Model: "resnet50_36cls"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 224, 224, 3)]     0         
                                                                 
 scale_to_255 (Rescaling)    (None, 224, 224, 3)       0         
                                                                 
 preprocess (Lambda)         (None, 224, 224, 3)       0         
                                                                 
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 gap (GlobalAveragePooling2  (None, 2048)              0         
 D)                                                              
                                                                 
 head_bn (BatchNormalizatio  (None, 2048)              8192      
 n)                                                 

## 6. Phase 1 - Train Head Only

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('=== Phase 1: Training Head Only ===')

history1 = model.fit(
    train_ds,
    epochs=EPOCHS_HEAD,
    verbose=1
)

=== Phase 1: Training Head Only ===
Epoch 1/10


98/98 [==============================] - 120s 1s/step - loss: 1.8019 - accuracy: 0.5294
Epoch 2/10
98/98 [==============================] - 133s 1s/step - loss: 0.6540 - accuracy: 0.8074
Epoch 3/10
98/98 [==============================] - 153s 2s/step - loss: 0.4239 - accuracy: 0.8668
Epoch 4/10
98/98 [==============================] - 150s 2s/step - loss: 0.3386 - accuracy: 0.8867
Epoch 5/10
98/98 [==============================] - 177s 2s/step - loss: 0.2691 - accuracy: 0.9101
Epoch 6/10
98/98 [==============================] - 176s 2s/step - loss: 0.2038 - accuracy: 0.9358
Epoch 7/10
98/98 [==============================] - 159s 2s/step - loss: 0.1796 - accuracy: 0.9390
Epoch 8/10
98/98 [==============================] - 118s 1s/step - loss: 0.1548 - accuracy: 0.9512
Epoch 9/10
98/98 [==============================] - 146s 1s/step - loss: 0.1461 - accuracy: 0.9554
Epoch 10/10
98/98 [==============================] - 139s 1s/step - los

## 7. Phase 2 - Fine-Tuning

In [8]:
# Unfreeze base model
base_model.trainable = True

# Keep lower layers frozen — only fine-tune deeper layers
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

print(f'Trainable layers : {sum(1 for l in base_model.layers if l.trainable)}')
print(f'Frozen layers    : {sum(1 for l in base_model.layers if not l.trainable)}')

Trainable layers : 35
Frozen layers    : 140


In [9]:
# Recompile with lower learning rate to fine-tune carefully
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('=== Phase 2: Fine-Tuning ===')

history2 = model.fit(
    train_ds,
    epochs=EPOCHS_FINE,
    verbose=1
)

=== Phase 2: Fine-Tuning ===
Epoch 1/20
98/98 [==============================] - 190s 2s/step - loss: 0.1055 - accuracy: 0.9682
Epoch 2/20
98/98 [==============================] - 193s 2s/step - loss: 0.0821 - accuracy: 0.9714
Epoch 3/20
98/98 [==============================] - 217s 2s/step - loss: 0.0734 - accuracy: 0.9798
Epoch 4/20
98/98 [==============================] - 202s 2s/step - loss: 0.0629 - accuracy: 0.9820
Epoch 5/20
98/98 [==============================] - 204s 2s/step - loss: 0.0564 - accuracy: 0.9846
Epoch 6/20
98/98 [==============================] - 189s 2s/step - loss: 0.0571 - accuracy: 0.9856
Epoch 7/20
98/98 [==============================] - 181s 2s/step - loss: 0.0465 - accuracy: 0.9875
Epoch 8/20
98/98 [==============================] - 199s 2s/step - loss: 0.0440 - accuracy: 0.9865
Epoch 9/20
98/98 [==============================] - 188s 2s/step - loss: 0.0371 - accuracy: 0.9900
Epoch 10/20
98/98 [==============================] - 180s 2s/step - loss: 0.0348

## 8. Training Accuracy

In [10]:
train_loss, train_acc = model.evaluate(train_ds, verbose=0)

print(f'Train Loss     : {train_loss:.4f}')
print(f'Train Accuracy : {train_acc:.4f} ({train_acc*100:.2f}%)')

Train Loss     : 0.0164
Train Accuracy : 0.9926 (99.26%)


## 9. Training Curves

In [ ]:
acc    = history1.history['accuracy'] + history2.history['accuracy']
loss   = history1.history['loss']     + history2.history['loss']
p1_end = len(history1.history['accuracy'])
epochs = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, acc)
axes[0].axvline(x=p1_end + 0.5, color='grey', linestyle='--', label='Phase 1 → 2')
axes[0].set_title('Training Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(epochs, loss)
axes[1].axvline(x=p1_end + 0.5, color='grey', linestyle='--', label='Phase 1 → 2')
axes[1].set_title('Training Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'resnet50_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Test Accuracy

In [12]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')

Test Loss     : 0.1041
Test Accuracy : 0.9721 (97.21%)


## 11. Classification Report

In [13]:
y_true = np.concatenate([np.argmax(lbl.numpy(), axis=1) for _, lbl in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

print('=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

=== Classification Report ===
               precision    recall  f1-score   support

        apple      0.889     0.800     0.842        10
       banana      1.000     0.778     0.875         9
     beetroot      1.000     1.000     1.000        10
  bell pepper      0.900     0.900     0.900        10
      cabbage      1.000     1.000     1.000        10
     capsicum      0.909     1.000     0.952        10
       carrot      1.000     1.000     1.000        10
  cauliflower      1.000     1.000     1.000        10
chilli pepper      0.909     1.000     0.952        10
         corn      0.889     0.800     0.842        10
     cucumber      1.000     1.000     1.000        10
     eggplant      1.000     1.000     1.000        10
       garlic      1.000     1.000     1.000        10
       ginger      1.000     1.000     1.000        10
       grapes      1.000     1.000     1.000        10
     jalepeno      1.000     1.000     1.000        10
         kiwi      1.000     1.000

## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.4, annot_kws={'size': 7}, ax=ax
)
ax.set(xlabel='Predicted', ylabel='True', title='Confusion Matrix - ResNet50')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'resnet50_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Save Model

In [15]:
model.save('resnet50_rawan.keras')
print('Model saved: resnet50_rawan.keras')

Model saved: resnet50_rawan.keras


## Summary

| Item | Value |
|------|-------|
| Architecture | ResNet50 (ImageNet weights) |
| Input | 224 × 224 × 3, float32 in [0, 1] |
| Preprocessing in model | Rescaling(×255) → resnet50.preprocess_input (BGR, mean-sub) |
| Classes | 36 |
| Head | GAP → BN → Dropout(0.3) → Dense(36, softmax) |
| Phase 1 | Frozen base, Adam(1e-3), 10 epochs |
| Phase 2 | Unfreeze layers 140+, Adam(1e-5), 20 epochs |
| Augmentation | flip, brightness, contrast, saturation |
| Test accuracy | **97.21%** |
| Saved model | `resnet50_rawan.keras` (accepts [0,1] input) |